In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RESUME TRAINING — EdgeNetV4 from epoch 100 → 200
# ════════════════════════════════════════════════════════════════════════
# Loads EdgeNet_V4_best.pth and continues training with the SAME setup:
#   - Same optimizer state (AdamW with parameter groups)
#   - Same scheduler state (CosineAnnealingWarmRestarts T_0=30 T_mult=2)
#   - Same EMA decay
#   - Same focal loss + class weights
#
# Why this should help:
#   - Best F1 (0.9541) was achieved on epoch 100 — still improving
#   - LR restart at epoch 90 gave +1.76% F1 in 10 epochs
#   - Next scheduled restart is at epoch 210
#   - Train F1 (0.91) < Val F1 (0.95) → no overfitting, room to grow
#
# Safety: original checkpoint untouched. New runs save to a separate file.
#         If new training degrades, just keep using EdgeNet_V4_best.pth.
# ════════════════════════════════════════════════════════════════════════

import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from sklearn.metrics import precision_recall_fscore_support
from pathlib import Path


# ── Config ──────────────────────────────────────────────────────────────
RESUME_PATH       = Path('/home/sufi/training_results/models/V4/EdgeNet_V4_best.pth')
V4_DIR            = Path('/home/sufi/training_results/models/V4')
NEW_BEST_PATH     = V4_DIR / 'EdgeNet_V4_resumed_best.pth'
METRICS_JSON_NEW  = V4_DIR / 'training_metrics_resumed.json'

START_EPOCH       = 101         # continue from where we left off
TARGET_EPOCH      = 220         # 10 epochs past the 210 restart
PATIENCE          = 40          # more patience since we're chasing restarts
LR_BACKBONE       = 1e-4
LR_CA             = 5e-4
LR_HEAD           = 1e-3
WEIGHT_DECAY      = 1e-4
GRAD_CLIP         = 1.0
EMA_DECAY         = 0.9995
FOCAL_GAMMA       = 2.0
MONITOR           = 'defect_f1'


# ════════════════════════════════════════════════════════════════════════
# 1. Load checkpoint into model
# ════════════════════════════════════════════════════════════════════════

print("="*70)
print("RESUMING EdgeNetV4 TRAINING")
print("="*70)

ckpt = torch.load(RESUME_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"\n✅ Loaded checkpoint")
print(f"   Source epoch  : {ckpt['epoch']}")
print(f"   Source F1     : {ckpt['score']:.4f}")
print(f"   Source bin    : {ckpt['val_metrics']['binary_acc']:.2f}%")
print(f"   Source def    : {ckpt['val_metrics']['defect_acc']:.2f}%")
print(f"   Source prod   : {ckpt['val_metrics']['product_acc']:.2f}%")


# ════════════════════════════════════════════════════════════════════════
# 2. Recompute class weights (in case data hasn't been touched)
# ════════════════════════════════════════════════════════════════════════

print("\nComputing class weights from training data...")
_cls_counts = torch.zeros(NUM_DEFECT_TYPES)
with torch.no_grad():
    for _, _, _def_lbl, _, _ in train_3head_loader:
        _valid = _def_lbl[_def_lbl != -1]
        for _i in range(NUM_DEFECT_TYPES):
            _cls_counts[_i] += (_valid == _i).sum()

_total       = _cls_counts.sum()
_cls_weights = _total / (_cls_counts.clamp(min=1) * NUM_DEFECT_TYPES)
_cls_weights = (_cls_weights / _cls_weights.mean()).to(device)

print(f"   Class weights:")
for i, name in enumerate(SEMANTIC_GROUPS):
    print(f"     [{i}] {name:<18} n={int(_cls_counts[i]):>5}  w={_cls_weights[i].item():.2f}")


# ════════════════════════════════════════════════════════════════════════
# 3. Focal Loss (same formulation as v4)
# ════════════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=-1):
        super().__init__()
        self.gamma        = gamma
        self.weight       = weight
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        valid = targets != self.ignore_index
        if valid.sum() == 0:
            return logits.sum() * 0.0
        t = targets[valid]
        l = logits[valid]
        ce_unweighted = F.cross_entropy(l, t, reduction='none')
        pt = torch.exp(-ce_unweighted)
        focal_factor = (1.0 - pt) ** self.gamma
        if self.weight is not None:
            cls_w = self.weight[t]
            return (focal_factor * cls_w * ce_unweighted).mean()
        return (focal_factor * ce_unweighted).mean()

focal_defect = FocalLoss(gamma=FOCAL_GAMMA,
                         weight=_cls_weights,
                         ignore_index=-1).to(device)


# ════════════════════════════════════════════════════════════════════════
# 4. Rebuild optimizer + scheduler at correct epoch position
# ════════════════════════════════════════════════════════════════════════

backbone_params = list(model.backbone.parameters())
ca_params       = (list(model.ca_mid.parameters()) +
                   list(model.ca_deep.parameters()))
head_params     = (list(model.neck.parameters())         +
                   list(model.binary_head.parameters())  +
                   list(model.product_head.parameters()) +
                   list(model.defect_head.parameters())  +
                   list(criterion_dict['multitask'].parameters()))

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LR_BACKBONE},
    {'params': ca_params,       'lr': LR_CA},
    {'params': head_params,     'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)

# Try to restore optimizer state (gives momentum continuity)
try:
    optimizer.load_state_dict(ckpt['optimizer'])
    print(f"\n✅ Restored optimizer state (momentum continuity preserved)")
except Exception as e:
    print(f"\n⚠️  Could not restore optimizer state — starting fresh: {e}")

# Rebuild scheduler — must be after optimizer.load_state_dict
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=30, T_mult=2, eta_min=1e-6
)

# Try to restore scheduler state (preserves restart schedule)
try:
    scheduler.load_state_dict(ckpt['scheduler'])
    print(f"✅ Restored scheduler state — next restart at epoch ~210")
except Exception as e:
    print(f"⚠️  Could not restore scheduler — fast-forwarding manually: {e}")
    # Fast-forward scheduler to epoch 100 if state didn't load
    for _ in range(START_EPOCH - 1):
        scheduler.step()

# Ensure multitask loss knows we're past warmup
criterion_dict['multitask'].set_epoch(START_EPOCH)


# ════════════════════════════════════════════════════════════════════════
# 5. EMA — initialize shadow from CURRENT (already-trained) weights
# ════════════════════════════════════════════════════════════════════════

class EMA:
    def __init__(self, model, decay=0.9995):
        self.decay  = decay
        self.shadow = {k: v.clone().detach()
                       for k, v in model.state_dict().items()}

    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(
                    v.detach(), alpha=1.0 - self.decay)

    def apply(self, model):
        self._backup = {k: v.clone() for k, v in model.state_dict().items()}
        for k, v in model.state_dict().items():
            v.copy_(self.shadow[k])

    def restore(self, model):
        for k, v in model.state_dict().items():
            v.copy_(self._backup[k])
        del self._backup

ema    = EMA(model, EMA_DECAY)
scaler = GradScaler('cuda')

print(f"✅ EMA shadow initialised from epoch {ckpt['epoch']} weights")
print(f"   (no random-init pollution since model is already trained)")


# ════════════════════════════════════════════════════════════════════════
# 6. Metrics + train/val functions (same logic as Cell 5)
# ════════════════════════════════════════════════════════════════════════

def compute_defect_metrics(preds, labels):
    mask = labels != -1
    if mask.sum() == 0:
        z = np.zeros(NUM_DEFECT_TYPES)
        return {'f1': 0.0, 'prec': 0.0, 'rec': 0.0,
                'per_p': z, 'per_r': z, 'per_f': z}
    p, r, f, _ = precision_recall_fscore_support(
        labels[mask], preds[mask], average=None,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    return {'f1': float(f.mean()), 'prec': float(p.mean()),
            'rec': float(r.mean()),
            'per_p': p, 'per_r': r, 'per_f': f}


def train_one_epoch(model, loader, optimizer, scaler, ema, epoch):
    model.train()
    criterion_dict['multitask'].set_epoch(epoch)
    tot = bin_c = def_c = def_tot = prod_c = 0
    all_pred, all_true = [], []

    for imgs, bin_lbl, def_lbl, prod_lbl, _ in loader:
        imgs     = imgs.to(device)
        bin_lbl  = bin_lbl.to(device)
        def_lbl  = def_lbl.to(device)
        prod_lbl = prod_lbl.to(device)

        optimizer.zero_grad()
        with autocast('cuda'):
            sb, sd, sp = model(imgs)
            sb_sq  = sb.squeeze(1)
            l_bin  = criterion_dict['binary'](sb_sq, bin_lbl.float())
            l_def  = focal_defect(sd, def_lbl)
            l_prod = criterion_dict['product'](sp, prod_lbl)
            tw     = criterion_dict['multitask'].weights()
            l_tot  = (float(tw[0]) * l_bin  +
                      float(tw[1]) * l_def  +
                      float(tw[2]) * l_prod)

        scaler.scale(l_tot).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)

        tot += imgs.size(0)
        with torch.no_grad():
            bin_c  += ((torch.sigmoid(sb_sq) > 0.5).long() == bin_lbl).sum().item()
            prod_c += (sp.argmax(1) == prod_lbl).sum().item()
            preds   = sd.argmax(1)
            mask    = def_lbl != -1
            if mask.sum() > 0:
                def_c   += (preds[mask] == def_lbl[mask]).sum().item()
                def_tot += mask.sum().item()
            all_pred.extend(preds.cpu().numpy())
            all_true.extend(def_lbl.cpu().numpy())

    m = compute_defect_metrics(np.array(all_pred), np.array(all_true))
    return {'binary_acc': 100.*bin_c/tot,
            'defect_acc': 100.*def_c/max(def_tot, 1),
            'product_acc': 100.*prod_c/tot,
            'train_f1':   m['f1'],
            'train_per_p': m['per_p'],
            'train_per_r': m['per_r'],
            'train_per_f': m['per_f']}


def validate_epoch(model, loader, ema, epoch):
    # EMA always active on resume — model is already trained
    ema.apply(model)
    model.eval()
    tot = bin_c = def_c = def_tot = prod_c = 0
    all_pred, all_true = [], []

    with torch.no_grad():
        for imgs, bin_lbl, def_lbl, prod_lbl, _ in loader:
            imgs     = imgs.to(device)
            bin_lbl  = bin_lbl.to(device)
            def_lbl  = def_lbl.to(device)
            prod_lbl = prod_lbl.to(device)
            with autocast('cuda'):
                sb, sd, sp = model(imgs)
                sb_sq      = sb.squeeze(1)
            tot    += imgs.size(0)
            bin_c  += ((torch.sigmoid(sb_sq) > 0.5).long() == bin_lbl).sum().item()
            prod_c += (sp.argmax(1) == prod_lbl).sum().item()
            preds   = sd.argmax(1)
            mask    = def_lbl != -1
            if mask.sum() > 0:
                def_c   += (preds[mask] == def_lbl[mask]).sum().item()
                def_tot += mask.sum().item()
            all_pred.extend(preds.cpu().numpy())
            all_true.extend(def_lbl.cpu().numpy())

    ema.restore(model)

    m = compute_defect_metrics(np.array(all_pred), np.array(all_true))
    return {'binary_acc': 100.*bin_c/tot,
            'defect_acc': 100.*def_c/max(def_tot, 1),
            'product_acc': 100.*prod_c/tot,
            'val_f1':   m['f1'],
            'val_per_p': m['per_p'],
            'val_per_r': m['per_r'],
            'val_per_f': m['per_f']}


# ════════════════════════════════════════════════════════════════════════
# 7. Resume Training Loop
# ════════════════════════════════════════════════════════════════════════

history       = []
best_score    = float(ckpt['score'])     # start from prior best (0.9541)
best_at_epoch = int(ckpt['epoch'])       # 100
no_improve    = 0
prev_per_f    = np.zeros(NUM_DEFECT_TYPES)

print(f"\n🚀 Resuming from epoch {START_EPOCH} → target {TARGET_EPOCH}")
print(f"   Prior best F1     : {best_score:.4f}  (epoch {best_at_epoch})")
print(f"   Patience          : {PATIENCE}")
print(f"   Monitor           : {MONITOR}")
print(f"   Next LR restart   : epoch ~210")
print(f"   New best saved to : {NEW_BEST_PATH}")
print(f"   Metrics JSON      : {METRICS_JSON_NEW}\n")

t0 = time.time()

for epoch in range(START_EPOCH, TARGET_EPOCH + 1):

    tr = train_one_epoch(model, train_3head_loader, optimizer, scaler, ema, epoch)
    vl = validate_epoch(model, val_3head_loader, ema, epoch)
    scheduler.step()

    score = vl['val_f1'] if MONITOR == 'defect_f1' else vl['defect_acc']
    per_p = vl['val_per_p']
    per_r = vl['val_per_r']
    per_f = vl['val_per_f']
    tw    = criterion_dict['multitask'].weights()
    lrs   = [f"{pg['lr']:.1e}" for pg in optimizer.param_groups]
    beat  = "  🏆 NEW BEST!" if score > best_score else ""
    r_tag = "  🔄 LR RESTART" if epoch == 210 else ""

    print(f"\nEpoch [{epoch:>3}/{TARGET_EPOCH}]  "
          f"LR={lrs}  "
          f"TaskW=[{' '.join(f'{float(w):.2f}' for w in tw)}]"
          f"{beat}{r_tag}")
    print(f"  Train → Bin={tr['binary_acc']:.1f}%  Def={tr['defect_acc']:.1f}%  "
          f"F1={tr['train_f1']:.3f}  Prod={tr['product_acc']:.1f}%")
    print(f"  Val   → Bin={vl['binary_acc']:.1f}%  Def={vl['defect_acc']:.1f}%  "
          f"F1={vl['val_f1']:.3f}  Prod={vl['product_acc']:.1f}%")
    print()
    print(f"  {'Class':<22} {'P':>6}  {'R':>6}  {'F1':>6}  Trend")
    print(f"  {'─'*50}")
    for i, name in enumerate(SEMANTIC_GROUPS):
        delta = per_f[i] - prev_per_f[i]
        arrow = '↑' if delta > 0.005 else ('↓' if delta < -0.005 else '→')
        watch = '  ◄ WATCH' if per_r[i] < 0.50 else ''
        print(f"  {name:<22} {per_p[i]:.3f}  {per_r[i]:.3f}  "
              f"{per_f[i]:.3f}  {arrow}{watch}")
    print(f"  {'─'*50}")
    print(f"  {'MACRO':<22} {per_p.mean():.3f}  "
          f"{per_r.mean():.3f}  {per_f.mean():.3f}")
    prev_per_f = per_f.copy()

    # ── Save new best ───────────────────────────────────────
    if score > best_score:
        best_score    = score
        best_at_epoch = epoch
        no_improve    = 0
        ema.apply(model)
        torch.save({
            'epoch':              epoch,
            'score':              best_score,
            'model_state_dict':   model.state_dict(),
            'optimizer':          optimizer.state_dict(),
            'scheduler':          scheduler.state_dict(),
            'val_metrics': {
                'binary_acc':      vl['binary_acc'],
                'defect_acc':      vl['defect_acc'],
                'defect_f1_macro': vl['val_f1'],
                'product_acc':     vl['product_acc'],
            },
            'defect_type_to_idx':  ckpt['defect_type_to_idx'],
            'product_type_to_idx': ckpt['product_type_to_idx'],
            'idx_to_defect_type':  ckpt['idx_to_defect_type'],
            'idx_to_product_type': ckpt['idx_to_product_type'],
            'resumed_from':        str(RESUME_PATH),
            'resumed_from_epoch':  int(ckpt['epoch']),
            'resumed_from_score':  float(ckpt['score']),
        }, NEW_BEST_PATH)
        ema.restore(model)
        print(f"\n  ✅ NEW BEST  {MONITOR}={best_score:.4f}  → saved {NEW_BEST_PATH.name}")
    else:
        no_improve += 1
        print(f"\n  (no improvement {no_improve}/{PATIENCE}  "
              f"— best={best_score:.4f} @ ep {best_at_epoch})")

    # ── Metrics JSON ─────────────────────────────────────────
    history.append({
        'epoch':          epoch,
        'train_f1':       tr['train_f1'],
        'train_def_acc':  tr['defect_acc'],
        'train_bin_acc':  tr['binary_acc'],
        'train_prod_acc': tr['product_acc'],
        'val_f1':         vl['val_f1'],
        'val_def_acc':    vl['defect_acc'],
        'val_bin_acc':    vl['binary_acc'],
        'val_prod_acc':   vl['product_acc'],
        'per_f':          per_f.tolist(),
        'per_p':          per_p.tolist(),
        'per_r':          per_r.tolist(),
        'best_score':     best_score,
        'best_at_epoch':  best_at_epoch,
        'lr_backbone':    optimizer.param_groups[0]['lr'],
        'lr_ca':          optimizer.param_groups[1]['lr'],
        'lr_head':        optimizer.param_groups[2]['lr'],
    })
    with open(METRICS_JSON_NEW, 'w') as f:
        json.dump({'class_names':   SEMANTIC_GROUPS,
                   'best_score':    best_score,
                   'best_at_epoch': best_at_epoch,
                   'resumed_from':  int(ckpt['epoch']),
                   'history':       history}, f, indent=2)

    # ── Early stopping ────────────────────────────────────────
    if no_improve >= PATIENCE:
        print(f"\n⏹  Early stopping at epoch {epoch}")
        print(f"   No improvement for {PATIENCE} epochs")
        break


# ════════════════════════════════════════════════════════════════════════
# 8. Final report
# ════════════════════════════════════════════════════════════════════════

elapsed = (time.time() - t0) / 60

print(f"\n{'='*70}")
print(f"RESUMED TRAINING COMPLETE")
print(f"{'='*70}")
print(f"  Time elapsed       : {elapsed:.1f} min")
print(f"  Epochs run         : {len(history)}")
print(f"  Started from       : F1={float(ckpt['score']):.4f}  (epoch {ckpt['epoch']})")
print(f"  Final best         : F1={best_score:.4f}  (epoch {best_at_epoch})")
print(f"  Improvement        : {best_score - float(ckpt['score']):+.4f}  "
      f"({(best_score - float(ckpt['score']))*100:+.2f}%)")
print()
if best_score > float(ckpt['score']):
    print(f"  ✅ Improved! New best: {NEW_BEST_PATH}")
else:
    print(f"  ⚠️  No improvement. Keep using original: {RESUME_PATH}")
print(f"{'='*70}")
print("✅ DONE")